# EFL Fantasy Points Prediction

Predicting next gameweek's points from player stats using regression models.

Data: 34 gameweeks of player performance data (2024-25 season)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# Seaborn defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Imports successful!')

## 1. Load & Combine Data

In [ ]:
# Load all gameweek CSVs
data_dir = Path('../data')
csv_files = sorted(data_dir.glob('player_stats_gw*.csv'))

dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)

print(f'Loaded {len(csv_files)} gameweeks')
print(f'Total rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'\nFirst few rows:')
df.head()

## 2. Data Exploration

In [ ]:
print('Data types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())
print(f'\nUnique positions: {df["position"].unique()}')
print(f'\nPoints statistics:')
print(df['points'].describe())

In [ ]:
# Points distribution by position
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.boxplot(column='points', by='position', ax=axes[0])
axes[0].set_title('Points Distribution by Position')
axes[0].set_ylabel('Points')

df['position'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Players per Position')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Make a copy for processing
df_ml = df.copy()

# Encode categorical features
le_pos = LabelEncoder()
df_ml['position_encoded'] = le_pos.fit_transform(df_ml['position'])

le_diff = LabelEncoder()
df_ml['fixture_difficulty_encoded'] = le_diff.fit_transform(df_ml['fixture_difficulty'].fillna('unknown'))

# Convert is_home to binary
df_ml['is_home_binary'] = df_ml['is_home'].map({'H': 1, 'A': 0})

print('Position encoding:')
for i, pos in enumerate(le_pos.classes_):
    print(f'  {pos}: {i}')

print('\nFixture difficulty encoding:')
for i, diff in enumerate(le_diff.classes_):
    print(f'  {diff}: {i}')

print('\nSample encoded data:')
print(df_ml[['position', 'position_encoded', 'fixture_difficulty', 'fixture_difficulty_encoded', 'is_home', 'is_home_binary']].head())

## 4. Build Training Data (with target shifted)

In [ ]:
# Sort by player and gameweek to enable shifting
df_ml = df_ml.sort_values(['player_id', 'gameweek']).reset_index(drop=True)

# For each player, shift points forward to create target (next GW points)
df_ml['next_gw_points'] = df_ml.groupby('player_id')['points'].shift(-1)

# Drop the last row of each player (no next GW)
df_ml = df_ml.dropna(subset=['next_gw_points'])

print(f'Rows after target creation: {len(df_ml):,}')
print(f'\nSample (player_id, gameweek, current points, next_gw_points):')
print(df_ml[['player_id', 'gameweek', 'points', 'next_gw_points']].head(10))

## 5. Select Features & Prepare Train/Test

In [ ]:
# Features: stats + fixture info + position + home/away
feature_cols = [
    'minutes_played', 'goals_scored', 'hat_tricks', 'assists', 'penalty_misses',
    'own_goals', 'yellow_cards', 'red_cards', 'saves', 'penalty_saves',
    'clean_sheet', 'goals_conceded', 'clearances', 'blocks', 'tackles',
    'interceptions', 'key_passes', 'shots_on_target',
    'position_encoded', 'fixture_difficulty_encoded', 'is_home_binary'
]

X = df_ml[feature_cols].fillna(0)
y = df_ml['next_gw_points']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeatures: {feature_cols}')

In [ ]:
# Split: use gameweeks as temporal split (no data leakage)
# Assume you want ~80% train, 20% test; adjust GW cutoff as needed

# Option 1: Time-based split by gameweek (recommended for time-series data)
gw_cutoff = 27  # Train on GW 1-27, test on GW 28-34
train_mask = df_ml['gameweek'] <= gw_cutoff

X_train = X[train_mask]
X_test = X[~train_mask]
y_train = y[train_mask]
y_test = y[~train_mask]

print(f'Train set: {len(X_train):,} samples (GW <= {gw_cutoff})')
print(f'Test set: {len(X_test):,} samples (GW > {gw_cutoff})')
print(f'\nTarget (points) ranges:')
print(f'  Train: {y_train.min():.0f} to {y_train.max():.0f}')
print(f'  Test: {y_test.min():.0f} to {y_test.max():.0f}')

## 6. Model 1: Linear Regression

In [ ]:
# Linear regression baseline
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print('Linear Regression:')
print(f'  MAE:  {mae_lr:.3f}')
print(f'  RMSE: {rmse_lr:.3f}')
print(f'  R²:   {r2_lr:.3f}')

## 7. Model 2: Random Forest

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print('Random Forest:')
print(f'  MAE:  {mae_rf:.3f}')
print(f'  RMSE: {rmse_rf:.3f}')
print(f'  R²:   {r2_rf:.3f}')

## 8. Model 3: XGBoost

In [ ]:
# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train, verbose=False)

y_pred_xgb = xgb_model.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print('XGBoost:')
print(f'  MAE:  {mae_xgb:.3f}')
print(f'  RMSE: {rmse_xgb:.3f}')
print(f'  R²:   {r2_xgb:.3f}')

## 9. Model Comparison

In [ ]:
# Summary comparison
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'MAE': [mae_lr, mae_rf, mae_xgb],
    'RMSE': [rmse_lr, rmse_rf, rmse_xgb],
    'R²': [r2_lr, r2_rf, r2_xgb]
})

print('\n=== MODEL COMPARISON ===')
print(results.to_string(index=False))
print('\n(Lower MAE/RMSE is better, higher R² is better)')

In [ ]:
# Visualize predictions vs actuals for best model
best_model_name = results.loc[results['MAE'].idxmin(), 'Model']
best_predictions = [y_pred_lr, y_pred_rf, y_pred_xgb][results['MAE'].argmin()]

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(y_test, best_predictions, alpha=0.5, s=20)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('Actual Points')
ax.set_ylabel('Predicted Points')
ax.set_title(f'{best_model_name}: Predicted vs Actual Points')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 10. Feature Importance (Random Forest)